# Milestone 2 — Hierarchical Parsing and Standardization

This notebook drives the **real** Milestone 2 pipeline in this repository
(`src/pipeline.py`).  It imports the parser and reference modules directly —
`hier_parser.parse_document`, `refs.extract_references` / `refs.unify_bibtex`,
`gather.gather_version` and `assemble.PaperAssembler` — so what you see here is
exactly what a full `python src/pipeline.py ...` run produces.

Pipeline: **gather → parse → references → unify + rename cites → assemble**.

Run it top to bottom. The heavy batch run is a separate CLI step (see the last
cell and the README, section 4).


In [ ]:
import json
import sys
import time
from collections import Counter
from pathlib import Path

# This notebook lives in src/; walk up to the repository root.
REPO = Path.cwd().resolve()
while not (REPO / "src" / "pipeline.py").exists() and REPO.parent != REPO:
    REPO = REPO.parent
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

from gather import gather_version
from hier_parser import parse_document
from assemble import iter_nodes, PaperAssembler
from refs import extract_references, unify_bibtex, rename_cites
import pipeline

print("REPO                 =", REPO)
print("parse_document       <-", parse_document.__module__)
print("extract_references   <-", extract_references.__module__)
print("unify_bibtex         <-", unify_bibtex.__module__)


In [ ]:
# Corpus: point at the Milestone 1 output (paper.json + v1/..vN/).
CORPUS = Path("corpus")                      # default from the README
if not CORPUS.exists():
    CORPUS = REPO.parent.parent / "2512.18231-2512.23230" / "corpus"   # local fallback
print("CORPUS =", CORPUS, "| exists =", CORPUS.is_dir())

papers = pipeline.list_papers(CORPUS)
print("publications:", len(papers))


In [ ]:
# Demo one publication end-to-end with the real modules.
paper_dir = papers[0]
paper = json.loads((paper_dir / "paper.json").read_text(encoding="utf-8"))
arxiv_id = paper["arxiv_id"]
print("publication:", arxiv_id, "| versions:", [v["version"] for v in paper["versions"]])

trees, bib_by_version = {}, {}
for info in sorted(paper["versions"], key=lambda v: v["version"]):
    number = info["version"]
    gathered = gather_version(paper_dir / f"v{number}")
    if gathered.status != "ok":
        print(f"  v{number}: {gathered.status}")
        continue
    trees[number] = parse_document(gathered.text)
    bib_by_version[number] = extract_references(gathered, paper_dir / f"v{number}")
    print(f"  v{number}: root={gathered.root}, nodes={sum(1 for _ in iter_nodes(trees[number]))}, "
          f"refs={len(bib_by_version[number])}")

# Show the tree of the first parsed version.
first = next(iter(trees))
def show(node, depth=0, limit=2, seen=set()):
    if depth > limit:
        return
    content = " ".join(node.content.split())[:80]
    print("  " * depth + f"- {node.kind}: {content}")
    for child in node.children[:8]:
        show(child, depth + 1, limit)
show(trees[first])


In [ ]:
# Reference deduplication across versions (unify + rename + assemble).
bibtex, rename_map = unify_bibtex(bib_by_version)
assembler = PaperAssembler(paper_dir.name, arxiv_id)
for number in sorted(trees):
    if rename_map:
        for node in iter_nodes(trees[number]):
            node.content = rename_cites(node.content, rename_map)
    assembler.add_version(number, trees[number])

problems = assembler.validate()
print("bibtex entries (deduplicated):", len(bibtex))
print("cite-key renames:", dict(list(rename_map.items())[:5]))
print("hierarchy problems:", problems[:3] if problems else "none")
print("elements after dedup:", len(assembler.element_rows()))
for entry in bibtex[:5]:
    print("  ", entry["citation_key"], "|", (entry.get("title") or "")[:50], "|", entry.get("year"))


In [ ]:
# Schema check of what the pipeline writes (parsed/<key>.json).
out_dir = Path("parsed")
out_dir.mkdir(exist_ok=True)
result = pipeline.process_paper((str(paper_dir), str(out_dir), 120, True))
data = json.loads((out_dir / f"{paper_dir.name}.json").read_text(encoding="utf-8"))

# elements: [element_id, kind, content]; hierarchy: [version, element_id, parent_id]
element_ids = {row[0] for row in data["elements"]}
kind_counts = Counter(row[1] for row in data["elements"])
missing = [row for row in data["hierarchy"] if row[1] not in element_ids or row[2] not in element_ids]
print("status:", result["status"])
print("kind distribution:", dict(kind_counts.most_common()))
print("hierarchy edges:", len(data["hierarchy"]), "| foreign-key violations:", len(missing))
print("bibtex rows:", len(data["bibtex"]))
print("stats:", data["stats"])


## Run the whole corpus

The batch runner (`src/pipeline.py`) is the real entry point and also produces
the report inputs (`parse_papers.csv`, `parse_versions.csv`, `parse_metrics.json`)
and the sharded Parquet tables:

```bash
python src/pipeline.py --corpus-dir corpus --out-dir parsed --work-dir work_m2 --limit 5 --force --workers 1   # smoke
python src/pipeline.py --corpus-dir corpus --out-dir parsed --work-dir work_m2 --workers 2                     # full
```

The report reads `work_m2/parse_metrics.json` for wall time / avg per paper /
peak RAM, and `work_m2/parse_papers.csv` for the per-publication success rate —
always with `pd.read_csv(..., dtype={"arxiv_id": str})`.
